## Working with SQL Databases

When working with SQL databases, it's important to understand the basic concepts and commands that allow you to interact with the data stored in these databases. Below are some key points to consider:

1. **Database Structure**: SQL databases are organized into tables, which consist of rows and columns. Each table represents a specific entity, and each row represents a record within that entity.

2. **Constraints**: Constraints are rules applied to columns in a table to enforce data integrity. Common constraints include PRIMARY KEY, FOREIGN KEY, UNIQUE, NOT NULL, and CHECK constraints.

3. **SQL Commands**: The most common SQL commands include:
   - `SELECT`: Used to retrieve data from one or more tables.
   - `INSERT`: Used to add new records to a table.
   - `UPDATE`: Used to modify existing records in a table.
   - `DELETE`: Used to remove records from a table.

4. **Data Types**: SQL supports various data types, such as integers, strings, dates, and booleans. It's important to choose the appropriate data type for each column to ensure data integrity and optimize performance.

5. **Indexes**: Indexes are used to speed up the retrieval of data from a table. They work by creating a data structure that allows for faster searches, but they can also slow down write operations, so it's important to use them judiciously.

6. **Joins**: Joins are used to combine data from multiple tables based on a related column. Common types of joins include INNER JOIN, LEFT JOIN, RIGHT JOIN, and FULL OUTER JOIN. Understanding how to use joins effectively is crucial for querying complex datasets.

7. **Transactions**: Transactions allow you to group multiple SQL commands into a single unit of work. This ensures that either all commands are executed successfully, or none are applied, maintaining data integrity. Transactions are typically managed using the `BEGIN`, `COMMIT`, and `ROLLBACK` commands.

8. **Normalization**: Normalization is the process of organizing data in a database to reduce redundancy and improve data integrity.

### The sqlite3 Module in Python

The `sqlite3` module in Python provides a lightweight disk-based database that doesn’t require a separate server process. It allows you to create, connect to, and interact with SQLite databases using SQL commands. Below are some key points about using the `sqlite3` module:

Here, we import the `sqlite3` module alongside the datetime module, which will be useful for handling date and time data in our database operations. 

The `sqlite3` module provides a straightforward way to work with SQLite databases, making it a popular choice for small to medium-sized applications.

In [1]:
import sqlite3
import random
from datetime import date, timedelta

### Creating and Connecting to a Database

Configuring the connection to the SQLite database is straightforward. You can use the `sqlite3.connect()` function to create a connection object, which allows you to execute SQL commands and manage transactions.

We will name our database file `company.db`. If this file does not exist, it will be created in the current working directory.

In [2]:
# Configuration
DATABASE = "company.db"
random.seed(42)

connection = sqlite3.connect(DATABASE)
cursor = connection.cursor()

You can drop the database file if you want to start fresh. This can be done using the `cursor.executescript()` method to execute SQL commands that drop tables or delete records. However, be cautious when dropping a database, as it will permanently remove all data stored in it.

In [3]:
cursor.executescript("""
DROP TABLE IF EXISTS Logins;
DROP TABLE IF EXISTS Sales;
DROP TABLE IF EXISTS DateRanges;
DROP TABLE IF EXISTS Products;
DROP TABLE IF EXISTS Customers;
DROP TABLE IF EXISTS Employees;
DROP TABLE IF EXISTS Departments;
""")

#### Creating Tables and Inserting Data

First, we create the Department table with columns for department ID and Name. The department ID is set as the primary key to ensure uniqueness.

Primary Key ultimately does not allow null or duplicate values in the column, ensuring that each department has a unique identifier.

In [4]:
cursor.execute("""
CREATE TABLE Departments (
    DepartmentID INTEGER PRIMARY KEY,
    DepartmentName TEXT NOT NULL UNIQUE
)
""")

departments = [
    (1, "IT"),
    (2, "Finance"),
    (3, "HR"),
    (4, "Marketing"),
    (5, "Sales"),
    (6, "Operations")
]

cursor.executemany("""
INSERT INTO Departments
(DepartmentID, DepartmentName)
VALUES (?, ?)
""", departments)

The Employee table is created with columns for employee ID, name, department ID, and hire date, etc. 

The employee ID is also set as the primary key, and the department ID is defined as a foreign key that references the Department table. This establishes a relationship between the two tables, ensuring that each employee belongs to a valid department.

In [5]:
cursor.execute("""
CREATE TABLE Employees (
    EmployeeID INTEGER PRIMARY KEY,
    FirstName TEXT NOT NULL,
    LastName TEXT NOT NULL,
    DepartmentID INTEGER,
    ManagerID INTEGER,
    Salary REAL NOT NULL,
    HireDate TEXT NOT NULL,
    BirthDate TEXT NOT NULL,
    Email TEXT UNIQUE,
    FOREIGN KEY (DepartmentID)
        REFERENCES Departments(DepartmentID),
    FOREIGN KEY (ManagerID)
        REFERENCES Employees(EmployeeID)
)
""")

For an introductory example, we will insert some sample data into both the Department and Employee tables. This will allow us to demonstrate how to query and manipulate the data in subsequent steps.

This can be done by using sampling techniques to generate random data for the tables. For example, we can create a few departments and employees with random names and hire dates.

In [6]:
first_names = [
    "Alice", "Bob", "Charlie", "Diana", "Ethan",
    "Fiona", "George", "Hannah", "Ian", "Julia",
    "Kevin", "Laura", "Michael", "Nina", "Oscar",
    "Paula", "Quinn", "Rachel", "Samuel", "Tina",
    "Victor", "Wendy", "Xavier", "Yuki", "Zach"
]

last_names = [
    "Anderson", "Brown", "Clark", "Davis", "Evans",
    "Foster", "Garcia", "Harris", "Irwin", "Johnson",
    "King", "Lewis", "Miller", "Nelson", "Owens",
    "Parker", "Quinn", "Roberts", "Smith", "Taylor",
    "Underwood", "Wilson", "Young", "Zhang", "White"
]

# Repeated salaries are intentional.
salary_values = [
    42000, 45000, 48000, 50000, 52000,
    55000, 58000, 60000, 62000, 65000,
    68000, 70000, 72000, 75000, 80000,
    85000, 90000, 95000, 100000, 110000
]

employees = []

start_hire = date(2018, 1, 1)

for i in range(1, 51):

    first = first_names[(i - 1) % len(first_names)]
    last = last_names[(i - 1) % len(last_names)]

    # Every 10th employee has no department.
    if i % 10 == 0:
        department_id = None
    else:
        department_id = ((i - 1) % 6) + 1

    # First 6 employees are managers.
    if i <= 6:
        manager_id = None
    else:
        manager_id = ((i - 1) % 6) + 1

    salary = random.choice(salary_values)

    # Some recent employees for the "last 6 months" question.
    if i in [45, 46, 47, 48, 49, 50]:
        hire_date = date(2026, random.randint(5, 9), random.randint(1, 28))
    else:
        days = random.randint(0, 3000)
        hire_date = start_hire + timedelta(days=days)

    # Birth dates between 1970 and 2000.
    birth_date = date(
        random.randint(1970, 2000),
        random.randint(1, 12),
        random.randint(1, 28)
    )

    email = f"{first.lower()}.{last.lower()}{i}@company.com"

    employees.append((
        i,
        first,
        last,
        department_id,
        manager_id,
        salary,
        hire_date.isoformat(),
        birth_date.isoformat(),
        email
    ))

Now, based on the employees data, we insert some sample records into the Employee table. This will help us demonstrate how to perform queries and retrieve data from the database.

In [7]:
cursor.executemany("""
INSERT INTO Employees (
    EmployeeID,
    FirstName,
    LastName,
    DepartmentID,
    ManagerID,
    Salary,
    HireDate,
    BirthDate,
    Email
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""", employees)

The Customers table is created with columns for customer ID, name, email, and country. The customer ID is set as the primary key to ensure uniqueness.

In [8]:
cursor.execute("""
CREATE TABLE Customers (
    CustomerID INTEGER PRIMARY KEY,
    FirstName TEXT NOT NULL,
    LastName TEXT NOT NULL,
    Email TEXT,
    Country TEXT
)
""")

Same with the Employee table, we also sample some data for the Customers table to demonstrate how to perform queries and retrieve data from the database.

In [9]:
customers = [
    (1, "John", "Adams", "john@example.com", "USA"),
    (2, "Mary", "Brown", "mary@example.com", "UK"),
    (3, "David", "Clark", "david@example.com", "Canada"),
    (4, "Emma", "Davis", "emma@example.com", "USA"),
    (5, "James", "Evans", "james@example.com", "Australia"),
    (6, "Sophia", "Foster", "sophia@example.com", "USA"),
    (7, "Daniel", "Garcia", "daniel@example.com", "Spain"),
    (8, "Olivia", "Harris", "olivia@example.com", "France"),
    (9, "William", "Irwin", "william@example.com", "USA"),
    (10, "Ava", "Johnson", "ava@example.com", "Germany"),
    (11, "Henry", "King", "henry@example.com", "Japan"),
    (12, "Mia", "Lewis", "mia@example.com", "USA"),
    (13, "Lucas", "Miller", "lucas@example.com", "Canada"),
    (14, "Isabella", "Nelson", "isabella@example.com", "UK"),
    (15, "Noah", "Owens", "noah@example.com", "USA")
]

cursor.executemany("""
INSERT INTO Customers
(CustomerID, FirstName, LastName, Email, Country)
VALUES (?, ?, ?, ?, ?)
""", customers)

Similarly we will create a Products table with columns for product ID, name, price, and stock quantity. The product ID will be set as the primary key to ensure uniqueness. The data is also sampled for the Products table to demonstrate how to perform queries and retrieve data from the database.

In [10]:
cursor.execute("""
CREATE TABLE Products (
    ProductID INTEGER PRIMARY KEY,
    ProductName TEXT NOT NULL,
    Category TEXT NOT NULL,
    Price REAL NOT NULL CHECK (Price > 0)
)
""")

products = [
    (1, "Laptop", "Electronics", 1200),
    (2, "Keyboard", "Electronics", 80),
    (3, "Mouse", "Electronics", 40),
    (4, "Monitor", "Electronics", 350),
    (5, "Headphones", "Electronics", 150),
    (6, "Office Chair", "Furniture", 300),
    (7, "Desk", "Furniture", 500),
    (8, "Notebook", "Stationery", 10),
    (9, "Backpack", "Accessories", 70),
    (10, "Webcam", "Electronics", 100)
]

cursor.executemany("""
INSERT INTO Products
(ProductID, ProductName, Category, Price)
VALUES (?, ?, ?, ?)
""", products)

We will create a Sales table with columns for sale ID, product ID, customer ID, quantity sold, and sale date. The sale ID will be set as the primary key to ensure uniqueness. The product ID and customer ID will be defined as foreign keys that reference the Products and Customers tables, respectively. This establishes relationships between the tables, ensuring that each sale is associated with a valid product and customer.

In [11]:
cursor.execute("""
CREATE TABLE Sales (
    SaleID INTEGER PRIMARY KEY,
    CustomerID INTEGER,
    ProductID INTEGER,
    SaleDate TEXT NOT NULL,
    Quantity INTEGER NOT NULL CHECK (Quantity > 0),
    UnitPrice REAL NOT NULL,
    FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID),
    FOREIGN KEY (ProductID) REFERENCES Products(ProductID)
)
""")

Sample data will be inserted into the Sales table to demonstrate how to perform queries and retrieve data from the database. This will allow us to analyze sales trends, customer behavior, and product performance.

In [12]:
sales = []

# Products 1-9 will be ordered.
# Product 10 intentionally receives no orders,
# allowing students to practice "products never ordered".

sale_id = 1

for i in range(100):

    customer_id = random.randint(1, 15)

    product_id = random.randint(1, 9)

    # Spread sales across 2026.
    month = random.randint(1, 10)
    day = random.randint(1, 28)

    sale_date = date(2026, month, day)

    quantity = random.randint(1, 5)

    # Find product price
    product_price = products[product_id - 1][3]

    sales.append((
        sale_id,
        customer_id,
        product_id,
        sale_date.isoformat(),
        quantity,
        product_price
    ))

    sale_id += 1

cursor.executemany("""
INSERT INTO Sales (
    SaleID,
    CustomerID,
    ProductID,
    SaleDate,
    Quantity,
    UnitPrice
)
VALUES (?, ?, ?, ?, ?, ?)
""", sales)

The Logins table is created with columns for login ID, username, and last login date. The login ID is set as the primary key to ensure uniqueness.

In [13]:
cursor.execute("""
CREATE TABLE Logins (
    LoginID INTEGER PRIMARY KEY,
    UserID INTEGER NOT NULL,
    LoginDate TEXT NOT NULL
)
""")

logins = []

login_id = 1

# Create login history for 10 users.
for user_id in range(1, 11):

    login_start = date(2026, 1, 1)

    for _ in range(random.randint(8, 15)):

        login_date = login_start + timedelta(
            days=random.randint(0, 280)
        )

        logins.append((
            login_id,
            user_id,
            login_date.isoformat()
        ))

        login_id += 1

cursor.executemany("""
INSERT INTO Logins
(LoginID, UserID, LoginDate)
VALUES (?, ?, ?)
""", logins)

The DateRange table is created with columns for range ID, start date, and end date. The range ID is set as the primary key to ensure uniqueness.

In [14]:
cursor.execute("""
CREATE TABLE DateRanges (
    RangeID INTEGER PRIMARY KEY,
    EmployeeID INTEGER,
    StartDate TEXT,
    EndDate TEXT
)
""")

date_ranges = [
    (1, 1, "2026-01-01", "2026-03-31"),
    (2, 2, "2026-02-15", "2026-05-15"),
    (3, 3, "2026-04-01", "2026-06-30"),
    (4, 4, "2026-05-01", "2026-08-31"),
    (5, 5, "2026-07-01", "2026-09-30"),
    (6, 6, "2026-08-15", "2026-10-31"),
    (7, 7, "2026-09-01", "2026-12-31"),
    (8, 8, "2026-03-01", "2026-04-30")
]

cursor.executemany("""
INSERT INTO DateRanges
(RangeID, EmployeeID, StartDate, EndDate)
VALUES (?, ?, ?, ?)
""", date_ranges)

Now all tables are created, we have to commit the changes to the database. This is done using the `conn.commit()` method, which saves all changes made during the current transaction.

Eventually, we close the connection to the database using the `conn.close()` method. This is an important step to free up resources and ensure that all changes are properly saved.

In [15]:
# Save everything
connection.commit()

# Display summary
tables = [
    "Departments",
    "Employees",
    "Customers",
    "Products",
    "Sales",
    "Logins",
    "DateRanges"
]

print("Database created successfully!")
print(f"Database: {DATABASE}")
print()

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table:15} : {count} records")

connection.close()

Database created successfully!
Database: company.db

Departments     : 6 records
Employees       : 50 records
Customers       : 15 records
Products        : 10 records
Sales           : 100 records
Logins          : 110 records
DateRanges      : 8 records


## Queries and Data Retrieval

Now that we have created the tables and inserted sample data, we can perform various queries to retrieve and analyze the data.

Some key methods:

- `cursor.execute()`: This method is used to execute SQL commands. You can pass a SQL query as a string to this method, and it will execute the command on the database.

- `fetchall()`: After executing a SELECT query, you can use this method to retrieve all the results as a list of tuples. Each tuple represents a row in the result set.

- `fetchone()`: If you only want to retrieve a single row from the result set, you can use this method. This will return the next row in the result set as a tuple.

- `fetchmany(size)`: This method allows you to retrieve a specified number of rows from the result set. You can pass an integer value to the size parameter, and it will return that many rows as a list of tuples.

In [16]:
# Reopen by creating a new connection instance
connection = sqlite3.connect(DATABASE)
cursor = connection.cursor()

# SQL Queries

Write your SQL query inside each cursor.execute() statement.

In [17]:
# 0. Select * for every database 
for table in tables:
    cursor.execute(f"SELECT * FROM {table}")
    print(f"\n{table}")
    
    # Retrieve and print the column names as a list
    column_names = [description[0] for description in cursor.description]
    print(f"Columns: {column_names}")
    
    for row in cursor.fetchmany(5):
        print(row)


Departments
Columns: ['DepartmentID', 'DepartmentName']
(1, 'IT')
(2, 'Finance')
(3, 'HR')
(4, 'Marketing')
(5, 'Sales')

Employees
Columns: ['EmployeeID', 'FirstName', 'LastName', 'DepartmentID', 'ManagerID', 'Salary', 'HireDate', 'BirthDate', 'Email']
(1, 'Alice', 'Anderson', 1, None, 50000.0, '2018-04-13', '1993-05-08', 'alice.anderson1@company.com')
(2, 'Bob', 'Brown', 2, None, 60000.0, '2019-07-26', '1993-02-22', 'bob.brown2@company.com')
(3, 'Charlie', 'Clark', 3, None, 95000.0, '2018-12-23', '1988-07-02', 'charlie.clark3@company.com')
(4, 'Diana', 'Davis', 4, None, 42000.0, '2019-01-19', '1976-04-17', 'diana.davis4@company.com')
(5, 'Ethan', 'Evans', 5, None, 110000.0, '2018-04-19', '1987-04-23', 'ethan.evans5@company.com')

Customers
Columns: ['CustomerID', 'FirstName', 'LastName', 'Email', 'Country']
(1, 'John', 'Adams', 'john@example.com', 'USA')
(2, 'Mary', 'Brown', 'mary@example.com', 'UK')
(3, 'David', 'Clark', 'david@example.com', 'Canada')
(4, 'Emma', 'Davis', 'emma@exa

In [18]:
# A reusable function to retrieve every found entry in an SQL query
def retrieve_information(cursor):
    # Fetch all duplicate records and store them in a variable
    rows = cursor.fetchall()

    # Check if the list is not empty
    if rows:
        print("Entry found!")
        for row in rows:
            print(row)
    else:
        print("Not found")


In [19]:
# 1. Find the second highest salary from the Employees table.
cursor.execute("""
SELECT * FROM Employees 
    WHERE Salary = (
    SELECT DISTINCT Salary 
    FROM Employees 
    ORDER BY Salary DESC 
    LIMIT 1 OFFSET 1
    );
""")

retrieve_information(cursor)

Entry found!
(24, 'Yuki', 'Zhang', 6, 6, 100000.0, '2026-01-19', '1980-04-21', 'yuki.zhang24@company.com')
(50, 'Zach', 'White', None, 2, 100000.0, '2026-06-01', '1972-12-21', 'zach.white50@company.com')


In [20]:
# 2. Find the duplicate records in a table.
'''
Here, we find duplicate customers
'''
cursor.execute("""
INSERT INTO Customers
(CustomerID, FirstName, LastName, Email, Country)
VALUES (20, 'Mary', 'Brown', 'mary@example.com', 'UK')
""")

cursor.execute("""
SELECT * FROM Customers WHERE Email IN (
    SELECT Email FROM Customers 
    GROUP BY Email 
    HAVING COUNT(*) > 1
    )
""")

# Fetch all duplicate records and store them in a variable
retrieve_information(cursor)

Entry found!
(2, 'Mary', 'Brown', 'mary@example.com', 'UK')
(20, 'Mary', 'Brown', 'mary@example.com', 'UK')


In [21]:
# 3. Remove duplicate records from a table.
cursor.execute("""
DELETE FROM Customers WHERE CustomerID NOT IN (
SELECT MIN(CustomerID) FROM Customers GROUP BY Email
)
""")

cursor.execute("""
SELECT * FROM Customers WHERE Email IN (
    SELECT Email FROM Customers 
    GROUP BY Email 
    HAVING COUNT(*) > 1
    )
""")

retrieve_information(cursor)

Not found


In [22]:
# 4. Find the top 5 highest salaries.
cursor.execute("""
SELECT EmployeeID, FirstName, LastName, DepartmentID, Salary
FROM Employees ORDER BY Salary DESC LIMIT 5
""")

retrieve_information(cursor)

Entry found!
(5, 'Ethan', 'Evans', 5, 110000.0)
(13, 'Michael', 'Miller', 1, 110000.0)
(32, 'George', 'Garcia', 2, 110000.0)
(46, 'Victor', 'Underwood', 4, 110000.0)
(24, 'Yuki', 'Zhang', 6, 100000.0)


In [23]:
# 5. Find the Nth highest salary.
def select_nth_salary(n):
    # Ensure n is valid (1st, 2nd, 3rd, etc.)
    if n < 1:
        print("Please enter a position greater than or equal to 1.")
        return None
    
    cursor = connection.cursor()

    # Subtract 1 because OFFSET is 0-indexed (e.g., 2nd salary needs OFFSET 1)
    cursor.execute("""
    SELECT DISTINCT Salary FROM Employees 
        ORDER BY Salary DESC LIMIT 1 OFFSET ?
    """, (n - 1, ))

    result = cursor.fetchone()
    
    if result:
        print(f"The {n}-nth highest salary is: {result[0]}")
    else:
        print(f"There is no {n}-nth highest salary.")


select_nth_salary(4)

The 4-nth highest salary is: 90000.0


In [24]:
# 6. Fetch departments and their total salary expenditure.
cursor.execute("""
SELECT DepartmentName, SUM(Salary) FROM Departments d JOIN Employees e 
    ON d.DepartmentID = e.DepartmentID
    GROUP BY DepartmentName
""")

retrieve_information(cursor)

Entry found!
('Finance', 458000.0)
('HR', 597000.0)
('IT', 616000.0)
('Marketing', 417000.0)
('Operations', 516000.0)
('Sales', 495000.0)


In [25]:
# 7. Find employees who joined in the last 6 months.

# Current Date - Hire Date ≤ 6 Months
# Months = [(Current Year - Hire Year) x 12] + (Current Month - Hire Month)
# The multiplication by 12 is used to convert the year difference into months.

cursor.execute("""
SELECT EmployeeID, FirstName, LastName, DepartmentID, Salary, HireDate 
FROM Employees WHERE 
(strftime('%Y', 'now') - strftime('%Y', HireDate)) * 12 
+ (strftime('%m', 'now') - strftime('%m', HireDate)) <= 6
""")

retrieve_information(cursor)

Entry found!
(45, 'Tina', 'Taylor', 3, 85000.0, '2026-09-06')
(46, 'Victor', 'Underwood', 4, 110000.0, '2026-08-07')
(47, 'Wendy', 'Wilson', 5, 58000.0, '2026-07-13')
(48, 'Xavier', 'Young', 6, 80000.0, '2026-09-15')
(49, 'Yuki', 'Zhang', 1, 48000.0, '2026-07-01')
(50, 'Zach', 'White', None, 100000.0, '2026-06-01')


In [26]:
# 8. Find employees who are not in any department.
cursor.execute("""
SELECT EmployeeID, FirstName, LastName, Salary FROM Employees 
WHERE DepartmentID IS NULL
""")

retrieve_information(cursor)

Entry found!
(10, 'Julia', 'Johnson', 70000.0)
(20, 'Tina', 'Taylor', 60000.0)
(30, 'Ethan', 'Evans', 45000.0)
(40, 'Oscar', 'Owens', 90000.0)
(50, 'Zach', 'White', 100000.0)


In [27]:
# 9. Find the total number of employees in each department.
cursor.execute("""
SELECT DepartmentName, COUNT(*) AS NumberofEmployees FROM Employees e 
JOIN Departments d ON e.DepartmentID = d.DepartmentID
GROUP BY e.DepartmentID
""")

retrieve_information(cursor)

Entry found!
('IT', 9)
('Finance', 7)
('HR', 8)
('Marketing', 6)
('Sales', 8)
('Operations', 7)


In [28]:
# 10. Pivot data to show department-wise employee count.
cursor.execute("""
    SELECT 
        SUM(CASE WHEN DepartmentName = 'IT' THEN 1 ELSE 0 END) AS HR,
        SUM(CASE WHEN DepartmentName = 'Finance' THEN 1 ELSE 0 END) AS Engineeringt,
        SUM(CASE WHEN DepartmentName = 'HR' THEN 1 ELSE 0 END) AS Sales,
        SUM(CASE WHEN DepartmentName = 'Marketing' THEN 1 ELSE 0 END) AS Marketing,
        SUM(CASE WHEN DepartmentName = 'Sales' THEN 1 ELSE 0 END) AS Marketing,
        SUM(CASE WHEN DepartmentName = 'Operations' THEN 1 ELSE 0 END) AS Operations
    FROM Employees e JOIN Departments d ON e.DepartmentID = d.DepartmentID;
""")

retrieve_information(cursor)

Entry found!
(9, 7, 8, 6, 8, 7)


In [29]:
# 11. Unpivot data from a table.
cursor.execute("""
SELECT DepartmentName, COUNT(*) AS NumberofEmployees FROM Employees e 
JOIN Departments d ON e.DepartmentID = d.DepartmentID
GROUP BY e.DepartmentID
""")

retrieve_information(cursor)

Entry found!
('IT', 9)
('Finance', 7)
('HR', 8)
('Marketing', 6)
('Sales', 8)
('Operations', 7)


In [30]:
# 12. Find month-over-month revenue growth.
cursor.execute("""
SELECT 
    strftime('%Y-%m', SaleDate) AS Month, 
    SUM(Quantity * UnitPrice) AS Revenue
FROM Sales
GROUP BY Month
""")

retrieve_information(cursor)

Entry found!
('2026-01', 6240.0)
('2026-02', 10170.0)
('2026-03', 6710.0)
('2026-04', 6120.0)
('2026-05', 10230.0)
('2026-06', 4950.0)
('2026-07', 7440.0)
('2026-08', 5120.0)
('2026-09', 7830.0)
('2026-10', 22000.0)


In [31]:
# 13. Find the running total of sales by date.
cursor.execute("""
SELECT 
    SaleDate, 
    SUM(Quantity * UnitPrice) AS DailyRevenue,
    SUM(SUM(Quantity * UnitPrice)) OVER (ORDER BY SaleDate) AS RunningTotal
FROM Sales
GROUP BY SaleDate
""")

retrieve_information(cursor)

Entry found!
('2026-01-06', 3500.0, 3500.0)
('2026-01-13', 30.0, 3530.0)
('2026-01-17', 2500.0, 6030.0)
('2026-01-24', 210.0, 6240.0)
('2026-02-02', 6200.0, 12440.0)
('2026-02-08', 1500.0, 13940.0)
('2026-02-20', 40.0, 13980.0)
('2026-02-23', 30.0, 14010.0)
('2026-02-25', 2400.0, 16410.0)
('2026-03-01', 350.0, 16760.0)
('2026-03-02', 350.0, 17110.0)
('2026-03-08', 160.0, 17270.0)
('2026-03-09', 210.0, 17480.0)
('2026-03-14', 140.0, 17620.0)
('2026-03-19', 450.0, 18070.0)
('2026-03-24', 350.0, 18420.0)
('2026-03-26', 700.0, 19120.0)
('2026-03-27', 2000.0, 21120.0)
('2026-03-28', 2000.0, 23120.0)
('2026-04-03', 160.0, 23280.0)
('2026-04-04', 500.0, 23780.0)
('2026-04-05', 500.0, 24280.0)
('2026-04-07', 40.0, 24320.0)
('2026-04-08', 150.0, 24470.0)
('2026-04-09', 2480.0, 26950.0)
('2026-04-14', 750.0, 27700.0)
('2026-04-17', 210.0, 27910.0)
('2026-04-22', 840.0, 28750.0)
('2026-04-26', 40.0, 28790.0)
('2026-04-27', 450.0, 29240.0)
('2026-05-02', 500.0, 29740.0)
('2026-05-04', 40.0, 29780.

In [32]:
# 14. Find gaps in a sequence of numbers.
cursor.execute("""
SELECT 
    EmployeeID, 
    (EmployeeID + 1) AS NextID
FROM Employees e1
WHERE NOT EXISTS (
    SELECT 1 FROM Employees e2 
    WHERE e2.EmployeeID = e1.EmployeeID + 1
)
""")

retrieve_information(cursor)

Entry found!
(50, 51)


In [33]:
# 15. Find the longest streak of consecutive login days for a user.
cursor.execute("""
WITH SequencedLogins AS (
    SELECT DISTINCT
        UserID,
        LoginDate,
        ROW_NUMBER() OVER (PARTITION BY UserID ORDER BY LoginDate) AS rn
    FROM Logins
),
StreakGroups AS (
    SELECT 
        UserID,
        COUNT(*) AS StreakLen
    FROM SequencedLogins
    -- julianday() converts dates to integers so we can subtract 'rn'
    GROUP BY UserID, (julianday(LoginDate) - rn)
)
SELECT 
    UserID, 
    MAX(StreakLen) AS LongestStreak 
FROM StreakGroups
GROUP BY UserID;

""")

retrieve_information(cursor)

Entry found!
(1, 2)
(2, 1)
(3, 1)
(4, 1)
(5, 3)
(6, 2)
(7, 1)
(8, 2)
(9, 1)
(10, 1)


In [34]:
# 16. Find the first login date of each user.
cursor.execute("""
SELECT UserID, MIN(LoginDate) AS FirstLoginDate
FROM Logins
GROUP BY UserID
""")

retrieve_information(cursor)

Entry found!
(1, '2026-01-10')
(2, '2026-02-25')
(3, '2026-02-08')
(4, '2026-01-07')
(5, '2026-02-24')
(6, '2026-01-15')
(7, '2026-01-06')
(8, '2026-01-09')
(9, '2026-04-24')
(10, '2026-03-03')


In [35]:
# 17. Find the last purchase of each customer.
cursor.execute("""
SELECT CustomerID, MAX(SaleDate) AS LastPurchaseDate
FROM Sales
GROUP BY CustomerID
""")

retrieve_information(cursor)

Entry found!
(1, '2026-10-19')
(2, '2026-10-07')
(3, '2026-08-16')
(4, '2026-10-28')
(5, '2026-09-22')
(6, '2026-05-18')
(7, '2026-09-15')
(8, '2026-10-07')
(9, '2026-09-01')
(10, '2026-08-15')
(11, '2026-10-11')
(12, '2026-10-26')
(13, '2026-10-27')
(14, '2026-09-24')
(15, '2026-10-24')


In [36]:
# 18. Find customers who bought in all months.
cursor.execute("""
SELECT CustomerID
FROM Sales
GROUP BY CustomerID
HAVING COUNT(DISTINCT strftime('%Y-%m', SaleDate)) = (
SELECT COUNT(DISTINCT strftime('%Y-%m', SaleDate)) FROM Sales
)
""")

retrieve_information(cursor)

Not found


In [37]:
# 19. Find products that were never ordered.
cursor.execute("""
SELECT ProductID, ProductName FROM Products
WHERE ProductID NOT IN (
    SELECT DISTINCT ProductID FROM Sales
)
""")

retrieve_information(cursor)

Entry found!
(10, 'Webcam')


In [38]:
# 20. Find employees earning more than the average salary.
cursor.execute("""
SELECT EmployeeID, FirstName, LastName, Salary FROM Employees
WHERE Salary > (SELECT AVG(Salary) FROM Employees)
""")

retrieve_information(cursor)

Entry found!
(3, 'Charlie', 'Clark', 95000.0)
(5, 'Ethan', 'Evans', 110000.0)
(6, 'Fiona', 'Foster', 95000.0)
(8, 'Hannah', 'Harris', 75000.0)
(10, 'Julia', 'Johnson', 70000.0)
(12, 'Laura', 'Lewis', 72000.0)
(13, 'Michael', 'Miller', 110000.0)
(17, 'Quinn', 'Quinn', 70000.0)
(21, 'Victor', 'Underwood', 95000.0)
(24, 'Yuki', 'Zhang', 100000.0)
(25, 'Zach', 'White', 85000.0)
(27, 'Bob', 'Brown', 95000.0)
(28, 'Charlie', 'Clark', 75000.0)
(31, 'Fiona', 'Foster', 75000.0)
(32, 'George', 'Garcia', 110000.0)
(34, 'Ian', 'Irwin', 95000.0)
(37, 'Laura', 'Lewis', 90000.0)
(40, 'Oscar', 'Owens', 90000.0)
(45, 'Tina', 'Taylor', 85000.0)
(46, 'Victor', 'Underwood', 110000.0)
(48, 'Xavier', 'Young', 80000.0)
(50, 'Zach', 'White', 100000.0)


In [39]:
# 22. Rank employees within each department by salary.
cursor.execute("""
SELECT 
    EmployeeID, 
    FirstName, 
    LastName, 
    DepartmentID, 
    Salary,
    RANK() OVER (PARTITION BY DepartmentID ORDER BY Salary DESC) AS SalaryRank
FROM Employees
""")

retrieve_information(cursor)

Entry found!
(50, 'Zach', 'White', None, 100000.0, 1)
(40, 'Oscar', 'Owens', None, 90000.0, 2)
(10, 'Julia', 'Johnson', None, 70000.0, 3)
(20, 'Tina', 'Taylor', None, 60000.0, 4)
(30, 'Ethan', 'Evans', None, 45000.0, 5)
(13, 'Michael', 'Miller', 1, 110000.0, 1)
(37, 'Laura', 'Lewis', 1, 90000.0, 2)
(25, 'Zach', 'White', 1, 85000.0, 3)
(31, 'Fiona', 'Foster', 1, 75000.0, 4)
(7, 'George', 'Garcia', 1, 62000.0, 5)
(1, 'Alice', 'Anderson', 1, 50000.0, 6)
(19, 'Samuel', 'Smith', 1, 48000.0, 7)
(43, 'Rachel', 'Roberts', 1, 48000.0, 7)
(49, 'Yuki', 'Zhang', 1, 48000.0, 7)
(32, 'George', 'Garcia', 2, 110000.0, 1)
(8, 'Hannah', 'Harris', 2, 75000.0, 2)
(38, 'Michael', 'Miller', 2, 65000.0, 3)
(2, 'Bob', 'Brown', 2, 60000.0, 4)
(26, 'Alice', 'Anderson', 2, 52000.0, 5)
(14, 'Nina', 'Nelson', 2, 48000.0, 6)
(44, 'Samuel', 'Smith', 2, 48000.0, 6)
(3, 'Charlie', 'Clark', 3, 95000.0, 1)
(21, 'Victor', 'Underwood', 3, 95000.0, 1)
(27, 'Bob', 'Brown', 3, 95000.0, 1)
(45, 'Tina', 'Taylor', 3, 85000.0, 4

In [40]:
# 23. Find the top 3 salaries in each department.
cursor.execute("""
WITH RankedEmployees AS (
    SELECT 
        EmployeeID, 
        FirstName, 
        LastName, 
        DepartmentID, 
        Salary,
        DENSE_RANK() OVER (PARTITION BY DepartmentID ORDER BY Salary DESC) AS SalaryRank
    FROM Employees
)
SELECT * 
FROM RankedEmployees
WHERE SalaryRank <= 3;
""")

retrieve_information(cursor)

Entry found!
(50, 'Zach', 'White', None, 100000.0, 1)
(40, 'Oscar', 'Owens', None, 90000.0, 2)
(10, 'Julia', 'Johnson', None, 70000.0, 3)
(13, 'Michael', 'Miller', 1, 110000.0, 1)
(37, 'Laura', 'Lewis', 1, 90000.0, 2)
(25, 'Zach', 'White', 1, 85000.0, 3)
(32, 'George', 'Garcia', 2, 110000.0, 1)
(8, 'Hannah', 'Harris', 2, 75000.0, 2)
(38, 'Michael', 'Miller', 2, 65000.0, 3)
(3, 'Charlie', 'Clark', 3, 95000.0, 1)
(21, 'Victor', 'Underwood', 3, 95000.0, 1)
(27, 'Bob', 'Brown', 3, 95000.0, 1)
(45, 'Tina', 'Taylor', 3, 85000.0, 2)
(9, 'Ian', 'Irwin', 3, 68000.0, 3)
(46, 'Victor', 'Underwood', 4, 110000.0, 1)
(34, 'Ian', 'Irwin', 4, 95000.0, 2)
(28, 'Charlie', 'Clark', 4, 75000.0, 3)
(5, 'Ethan', 'Evans', 5, 110000.0, 1)
(17, 'Quinn', 'Quinn', 5, 70000.0, 2)
(23, 'Xavier', 'Young', 5, 68000.0, 3)
(24, 'Yuki', 'Zhang', 6, 100000.0, 1)
(6, 'Fiona', 'Foster', 6, 95000.0, 2)
(48, 'Xavier', 'Young', 6, 80000.0, 3)


In [41]:
# 24. Find common records between two tables.
cursor.execute("""
SELECT * FROM Employees e
WHERE EXISTS (
    SELECT 1 FROM Departments d 
    WHERE e.DepartmentID = d.DepartmentID
)
""")

retrieve_information(cursor)

Entry found!
(1, 'Alice', 'Anderson', 1, None, 50000.0, '2018-04-13', '1993-05-08', 'alice.anderson1@company.com')
(2, 'Bob', 'Brown', 2, None, 60000.0, '2019-07-26', '1993-02-22', 'bob.brown2@company.com')
(3, 'Charlie', 'Clark', 3, None, 95000.0, '2018-12-23', '1988-07-02', 'charlie.clark3@company.com')
(4, 'Diana', 'Davis', 4, None, 42000.0, '2019-01-19', '1976-04-17', 'diana.davis4@company.com')
(5, 'Ethan', 'Evans', 5, None, 110000.0, '2018-04-19', '1987-04-23', 'ethan.evans5@company.com')
(6, 'Fiona', 'Foster', 6, None, 95000.0, '2022-09-15', '1977-08-19', 'fiona.foster6@company.com')
(7, 'George', 'Garcia', 1, 1, 62000.0, '2018-01-27', '1994-03-23', 'george.garcia7@company.com')
(8, 'Hannah', 'Harris', 2, 2, 75000.0, '2021-10-25', '1978-03-07', 'hannah.harris8@company.com')
(9, 'Ian', 'Irwin', 3, 3, 68000.0, '2019-02-23', '1972-07-04', 'ian.irwin9@company.com')
(11, 'Kevin', 'King', 5, 5, 45000.0, '2026-03-08', '1984-09-04', 'kevin.king11@company.com')
(12, 'Laura', 'Lewis', 6, 

In [42]:
# 25. Find records that are in one table but not in another.
cursor.execute("""
SELECT * FROM Employees e
WHERE NOT EXISTS (
    SELECT 1 FROM Departments d 
    WHERE e.DepartmentID = d.DepartmentID
)
""")

retrieve_information(cursor)

Entry found!
(10, 'Julia', 'Johnson', None, 4, 70000.0, '2021-11-09', '1989-05-26', 'julia.johnson10@company.com')
(20, 'Tina', 'Taylor', None, 2, 60000.0, '2019-11-01', '1984-07-09', 'tina.taylor20@company.com')
(30, 'Ethan', 'Evans', None, 6, 45000.0, '2019-03-26', '1974-11-06', 'ethan.evans30@company.com')
(40, 'Oscar', 'Owens', None, 4, 90000.0, '2018-01-03', '1989-06-16', 'oscar.owens40@company.com')
(50, 'Zach', 'White', None, 2, 100000.0, '2026-06-01', '1972-12-21', 'zach.white50@company.com')


In [43]:
# 26. Split a full name into first name and last name.
cursor.execute("""
SELECT 
    EmployeeID, 
    FirstName || ' ' || LastName AS FullName,
    FirstName,
    LastName
FROM Employees
""")

retrieve_information(cursor)

Entry found!
(1, 'Alice Anderson', 'Alice', 'Anderson')
(2, 'Bob Brown', 'Bob', 'Brown')
(3, 'Charlie Clark', 'Charlie', 'Clark')
(4, 'Diana Davis', 'Diana', 'Davis')
(5, 'Ethan Evans', 'Ethan', 'Evans')
(6, 'Fiona Foster', 'Fiona', 'Foster')
(7, 'George Garcia', 'George', 'Garcia')
(8, 'Hannah Harris', 'Hannah', 'Harris')
(9, 'Ian Irwin', 'Ian', 'Irwin')
(10, 'Julia Johnson', 'Julia', 'Johnson')
(11, 'Kevin King', 'Kevin', 'King')
(12, 'Laura Lewis', 'Laura', 'Lewis')
(13, 'Michael Miller', 'Michael', 'Miller')
(14, 'Nina Nelson', 'Nina', 'Nelson')
(15, 'Oscar Owens', 'Oscar', 'Owens')
(16, 'Paula Parker', 'Paula', 'Parker')
(17, 'Quinn Quinn', 'Quinn', 'Quinn')
(18, 'Rachel Roberts', 'Rachel', 'Roberts')
(19, 'Samuel Smith', 'Samuel', 'Smith')
(20, 'Tina Taylor', 'Tina', 'Taylor')
(21, 'Victor Underwood', 'Victor', 'Underwood')
(22, 'Wendy Wilson', 'Wendy', 'Wilson')
(23, 'Xavier Young', 'Xavier', 'Young')
(24, 'Yuki Zhang', 'Yuki', 'Zhang')
(25, 'Zach White', 'Zach', 'White')
(26, '

In [44]:
# 27. Concatenate first name and last name into a full name.
cursor.execute("""
SELECT 
    EmployeeID, 
    FirstName || ' ' || LastName AS FullName
FROM Employees
""")
retrieve_information(cursor)

Entry found!
(1, 'Alice Anderson')
(2, 'Bob Brown')
(3, 'Charlie Clark')
(4, 'Diana Davis')
(5, 'Ethan Evans')
(6, 'Fiona Foster')
(7, 'George Garcia')
(8, 'Hannah Harris')
(9, 'Ian Irwin')
(10, 'Julia Johnson')
(11, 'Kevin King')
(12, 'Laura Lewis')
(13, 'Michael Miller')
(14, 'Nina Nelson')
(15, 'Oscar Owens')
(16, 'Paula Parker')
(17, 'Quinn Quinn')
(18, 'Rachel Roberts')
(19, 'Samuel Smith')
(20, 'Tina Taylor')
(21, 'Victor Underwood')
(22, 'Wendy Wilson')
(23, 'Xavier Young')
(24, 'Yuki Zhang')
(25, 'Zach White')
(26, 'Alice Anderson')
(27, 'Bob Brown')
(28, 'Charlie Clark')
(29, 'Diana Davis')
(30, 'Ethan Evans')
(31, 'Fiona Foster')
(32, 'George Garcia')
(33, 'Hannah Harris')
(34, 'Ian Irwin')
(35, 'Julia Johnson')
(36, 'Kevin King')
(37, 'Laura Lewis')
(38, 'Michael Miller')
(39, 'Nina Nelson')
(40, 'Oscar Owens')
(41, 'Paula Parker')
(42, 'Quinn Quinn')
(43, 'Rachel Roberts')
(44, 'Samuel Smith')
(45, 'Tina Taylor')
(46, 'Victor Underwood')
(47, 'Wendy Wilson')
(48, 'Xavier Yo

In [45]:
# 28. Extract year, month, and day from a date column.
cursor.execute("""
SELECT 
    EmployeeID, 
    HireDate,
    strftime('%Y', HireDate) AS Year,
    strftime('%m', HireDate) AS Month,
    strftime('%d', HireDate) AS Day
FROM Employees
""")

retrieve_information(cursor)

Entry found!
(1, '2018-04-13', '2018', '04', '13')
(2, '2019-07-26', '2019', '07', '26')
(3, '2018-12-23', '2018', '12', '23')
(4, '2019-01-19', '2019', '01', '19')
(5, '2018-04-19', '2018', '04', '19')
(6, '2022-09-15', '2022', '09', '15')
(7, '2018-01-27', '2018', '01', '27')
(8, '2021-10-25', '2021', '10', '25')
(9, '2019-02-23', '2019', '02', '23')
(10, '2021-11-09', '2021', '11', '09')
(11, '2026-03-08', '2026', '03', '08')
(12, '2018-11-19', '2018', '11', '19')
(13, '2022-01-21', '2022', '01', '21')
(14, '2018-07-07', '2018', '07', '07')
(15, '2018-11-23', '2018', '11', '23')
(16, '2022-04-06', '2022', '04', '06')
(17, '2019-10-29', '2019', '10', '29')
(18, '2025-11-14', '2025', '11', '14')
(19, '2024-10-31', '2024', '10', '31')
(20, '2019-11-01', '2019', '11', '01')
(21, '2020-06-18', '2020', '06', '18')
(22, '2020-07-27', '2020', '07', '27')
(23, '2022-07-02', '2022', '07', '02')
(24, '2026-01-19', '2026', '01', '19')
(25, '2022-06-09', '2022', '06', '09')
(26, '2020-12-20', '2

In [46]:
# 29. Calculate age from date of birth.
cursor.execute("""
SELECT 
    EmployeeID, 
    BirthDate,
    (strftime('%Y', 'now') - strftime('%Y', BirthDate)) - 
    (strftime('%m-%d', 'now') < strftime('%m-%d', BirthDate)) AS Age
FROM Employees
""")

retrieve_information(cursor)

Entry found!
(1, '1993-05-08', 33)
(2, '1993-02-22', 33)
(3, '1988-07-02', 38)
(4, '1976-04-17', 50)
(5, '1987-04-23', 39)
(6, '1977-08-19', 49)
(7, '1994-03-23', 32)
(8, '1978-03-07', 48)
(9, '1972-07-04', 54)
(10, '1989-05-26', 37)
(11, '1984-09-04', 42)
(12, '1987-05-27', 39)
(13, '1988-04-23', 38)
(14, '1991-04-25', 35)
(15, '1997-04-28', 29)
(16, '1978-08-21', 48)
(17, '1981-06-07', 45)
(18, '1999-11-21', 26)
(19, '1990-03-18', 36)
(20, '1984-07-09', 42)
(21, '1991-06-27', 35)
(22, '1996-01-26', 30)
(23, '1978-02-07', 48)
(24, '1980-04-21', 46)
(25, '1998-11-15', 27)
(26, '1974-04-24', 52)
(27, '1978-12-19', 47)
(28, '1982-06-08', 44)
(29, '1985-02-25', 41)
(30, '1974-11-06', 51)
(31, '1972-07-13', 54)
(32, '1986-05-18', 40)
(33, '1993-02-22', 33)
(34, '1994-11-11', 31)
(35, '1983-03-15', 43)
(36, '1998-12-09', 27)
(37, '1986-02-28', 40)
(38, '1986-10-07', 40)
(39, '1994-03-18', 32)
(40, '1989-06-16', 37)
(41, '1999-06-27', 27)
(42, '1971-04-19', 55)
(43, '1993-08-27', 33)
(44, '1

In [47]:
# 30. Find the number of working days in a month.
cursor.execute("""
SELECT 
    strftime('%Y-%m', SaleDate) AS Month,
    COUNT(DISTINCT SaleDate) AS WorkingDays
FROM Sales
GROUP BY Month
""")

retrieve_information(cursor)

Entry found!
('2026-01', 4)
('2026-02', 5)
('2026-03', 10)
('2026-04', 11)
('2026-05', 13)
('2026-06', 7)
('2026-07', 8)
('2026-08', 9)
('2026-09', 8)
('2026-10', 12)


In [48]:
# 31. Find employees with the same salary.
cursor.execute("""
SELECT EmployeeID, FirstName, LastName, Salary
FROM Employees
WHERE Salary IN (
    SELECT Salary FROM Employees 
    GROUP BY Salary 
    HAVING COUNT(*) > 1
)
""")

retrieve_information(cursor)

Entry found!
(1, 'Alice', 'Anderson', 50000.0)
(2, 'Bob', 'Brown', 60000.0)
(3, 'Charlie', 'Clark', 95000.0)
(4, 'Diana', 'Davis', 42000.0)
(5, 'Ethan', 'Evans', 110000.0)
(6, 'Fiona', 'Foster', 95000.0)
(7, 'George', 'Garcia', 62000.0)
(8, 'Hannah', 'Harris', 75000.0)
(9, 'Ian', 'Irwin', 68000.0)
(10, 'Julia', 'Johnson', 70000.0)
(11, 'Kevin', 'King', 45000.0)
(13, 'Michael', 'Miller', 110000.0)
(14, 'Nina', 'Nelson', 48000.0)
(15, 'Oscar', 'Owens', 65000.0)
(16, 'Paula', 'Parker', 50000.0)
(17, 'Quinn', 'Quinn', 70000.0)
(18, 'Rachel', 'Roberts', 62000.0)
(19, 'Samuel', 'Smith', 48000.0)
(20, 'Tina', 'Taylor', 60000.0)
(21, 'Victor', 'Underwood', 95000.0)
(22, 'Wendy', 'Wilson', 45000.0)
(23, 'Xavier', 'Young', 68000.0)
(24, 'Yuki', 'Zhang', 100000.0)
(25, 'Zach', 'White', 85000.0)
(26, 'Alice', 'Anderson', 52000.0)
(27, 'Bob', 'Brown', 95000.0)
(28, 'Charlie', 'Clark', 75000.0)
(29, 'Diana', 'Davis', 52000.0)
(30, 'Ethan', 'Evans', 45000.0)
(31, 'Fiona', 'Foster', 75000.0)
(32, 'Geo

In [49]:
# 32. Find the highest-paid employee in each department.
cursor.execute("""
SELECT 
    EmployeeID, 
    FirstName, 
    LastName, 
    DepartmentID, 
    Salary
FROM Employees e1
WHERE Salary = (
    SELECT MAX(Salary) FROM Employees e2 
    WHERE e1.DepartmentID = e2.DepartmentID
)
""")

retrieve_information(cursor)

Entry found!
(3, 'Charlie', 'Clark', 3, 95000.0)
(5, 'Ethan', 'Evans', 5, 110000.0)
(13, 'Michael', 'Miller', 1, 110000.0)
(21, 'Victor', 'Underwood', 3, 95000.0)
(24, 'Yuki', 'Zhang', 6, 100000.0)
(27, 'Bob', 'Brown', 3, 95000.0)
(32, 'George', 'Garcia', 2, 110000.0)
(46, 'Victor', 'Underwood', 4, 110000.0)


In [50]:
# 33. Find all managers and their reporting employees.
cursor.execute("""
SELECT 
    m.EmployeeID AS ManagerID, 
    m.FirstName || ' ' || m.LastName AS ManagerName,
    e.EmployeeID AS EmployeeID,
    e.FirstName || ' ' || e.LastName AS EmployeeName
FROM Employees m
LEFT JOIN Employees e ON m.EmployeeID = e.ManagerID
WHERE m.EmployeeID IN (SELECT DISTINCT ManagerID FROM Employees WHERE ManagerID IS NOT NULL)
""")

retrieve_information(cursor)

Entry found!
(1, 'Alice Anderson', 7, 'George Garcia')
(1, 'Alice Anderson', 13, 'Michael Miller')
(1, 'Alice Anderson', 19, 'Samuel Smith')
(1, 'Alice Anderson', 25, 'Zach White')
(1, 'Alice Anderson', 31, 'Fiona Foster')
(1, 'Alice Anderson', 37, 'Laura Lewis')
(1, 'Alice Anderson', 43, 'Rachel Roberts')
(1, 'Alice Anderson', 49, 'Yuki Zhang')
(2, 'Bob Brown', 8, 'Hannah Harris')
(2, 'Bob Brown', 14, 'Nina Nelson')
(2, 'Bob Brown', 20, 'Tina Taylor')
(2, 'Bob Brown', 26, 'Alice Anderson')
(2, 'Bob Brown', 32, 'George Garcia')
(2, 'Bob Brown', 38, 'Michael Miller')
(2, 'Bob Brown', 44, 'Samuel Smith')
(2, 'Bob Brown', 50, 'Zach White')
(3, 'Charlie Clark', 9, 'Ian Irwin')
(3, 'Charlie Clark', 15, 'Oscar Owens')
(3, 'Charlie Clark', 21, 'Victor Underwood')
(3, 'Charlie Clark', 27, 'Bob Brown')
(3, 'Charlie Clark', 33, 'Hannah Harris')
(3, 'Charlie Clark', 39, 'Nina Nelson')
(3, 'Charlie Clark', 45, 'Tina Taylor')
(4, 'Diana Davis', 10, 'Julia Johnson')
(4, 'Diana Davis', 16, 'Paula Par

In [51]:
# 34. Reverse a string.
def reverse_string(s):
    return s[::-1]

connection.create_function("reverse_string", 1, reverse_string)

cursor.execute("""
SELECT 
    EmployeeID, 
    FirstName, 
    LastName,
    reverse_string(FirstName || ' ' || LastName) AS ReversedFullName
FROM Employees
""")

retrieve_information(cursor)

Entry found!
(1, 'Alice', 'Anderson', 'nosrednA ecilA')
(2, 'Bob', 'Brown', 'nworB boB')
(3, 'Charlie', 'Clark', 'kralC eilrahC')
(4, 'Diana', 'Davis', 'sivaD anaiD')
(5, 'Ethan', 'Evans', 'snavE nahtE')
(6, 'Fiona', 'Foster', 'retsoF anoiF')
(7, 'George', 'Garcia', 'aicraG egroeG')
(8, 'Hannah', 'Harris', 'sirraH hannaH')
(9, 'Ian', 'Irwin', 'niwrI naI')
(10, 'Julia', 'Johnson', 'nosnhoJ ailuJ')
(11, 'Kevin', 'King', 'gniK niveK')
(12, 'Laura', 'Lewis', 'siweL aruaL')
(13, 'Michael', 'Miller', 'relliM leahciM')
(14, 'Nina', 'Nelson', 'nosleN aniN')
(15, 'Oscar', 'Owens', 'snewO racsO')
(16, 'Paula', 'Parker', 'rekraP aluaP')
(17, 'Quinn', 'Quinn', 'nniuQ nniuQ')
(18, 'Rachel', 'Roberts', 'streboR lehcaR')
(19, 'Samuel', 'Smith', 'htimS leumaS')
(20, 'Tina', 'Taylor', 'rolyaT aniT')
(21, 'Victor', 'Underwood', 'doowrednU rotciV')
(22, 'Wendy', 'Wilson', 'nosliW ydneW')
(23, 'Xavier', 'Young', 'gnuoY reivaX')
(24, 'Yuki', 'Zhang', 'gnahZ ikuY')
(25, 'Zach', 'White', 'etihW hcaZ')
(26, '

In [52]:
# 35. Find the frequency of each word in a text column.
cursor.execute("""
SELECT 
    ProductName,
    LENGTH(ProductName) - LENGTH(REPLACE(ProductName, ' ', '')) + 1 AS WordCount
FROM Products
""")

retrieve_information(cursor)

Entry found!
('Laptop', 1)
('Keyboard', 1)
('Mouse', 1)
('Monitor', 1)
('Headphones', 1)
('Office Chair', 2)
('Desk', 1)
('Notebook', 1)
('Backpack', 1)
('Webcam', 1)


In [53]:
# 36. Find rows where a column is NULL.
cursor.execute("""
SELECT * FROM Employees WHERE DepartmentID IS NULL
""")

retrieve_information(cursor)

Entry found!
(10, 'Julia', 'Johnson', None, 4, 70000.0, '2021-11-09', '1989-05-26', 'julia.johnson10@company.com')
(20, 'Tina', 'Taylor', None, 2, 60000.0, '2019-11-01', '1984-07-09', 'tina.taylor20@company.com')
(30, 'Ethan', 'Evans', None, 6, 45000.0, '2019-03-26', '1974-11-06', 'ethan.evans30@company.com')
(40, 'Oscar', 'Owens', None, 4, 90000.0, '2018-01-03', '1989-06-16', 'oscar.owens40@company.com')
(50, 'Zach', 'White', None, 2, 100000.0, '2026-06-01', '1972-12-21', 'zach.white50@company.com')


In [54]:
# 37. Replace NULL values with a default value.
cursor.execute("""
SELECT 
    EmployeeID, 
    FirstName, 
    LastName, 
    COALESCE(DepartmentID, 'No Department') AS DepartmentID
FROM Employees
""")

retrieve_information(cursor)

Entry found!
(1, 'Alice', 'Anderson', 1)
(2, 'Bob', 'Brown', 2)
(3, 'Charlie', 'Clark', 3)
(4, 'Diana', 'Davis', 4)
(5, 'Ethan', 'Evans', 5)
(6, 'Fiona', 'Foster', 6)
(7, 'George', 'Garcia', 1)
(8, 'Hannah', 'Harris', 2)
(9, 'Ian', 'Irwin', 3)
(10, 'Julia', 'Johnson', 'No Department')
(11, 'Kevin', 'King', 5)
(12, 'Laura', 'Lewis', 6)
(13, 'Michael', 'Miller', 1)
(14, 'Nina', 'Nelson', 2)
(15, 'Oscar', 'Owens', 3)
(16, 'Paula', 'Parker', 4)
(17, 'Quinn', 'Quinn', 5)
(18, 'Rachel', 'Roberts', 6)
(19, 'Samuel', 'Smith', 1)
(20, 'Tina', 'Taylor', 'No Department')
(21, 'Victor', 'Underwood', 3)
(22, 'Wendy', 'Wilson', 4)
(23, 'Xavier', 'Young', 5)
(24, 'Yuki', 'Zhang', 6)
(25, 'Zach', 'White', 1)
(26, 'Alice', 'Anderson', 2)
(27, 'Bob', 'Brown', 3)
(28, 'Charlie', 'Clark', 4)
(29, 'Diana', 'Davis', 5)
(30, 'Ethan', 'Evans', 'No Department')
(31, 'Fiona', 'Foster', 1)
(32, 'George', 'Garcia', 2)
(33, 'Hannah', 'Harris', 3)
(34, 'Ian', 'Irwin', 4)
(35, 'Julia', 'Johnson', 5)
(36, 'Kevin', 'K

In [55]:
# 38. Trim spaces from the beginning and end of a string.
cursor.execute("""
SELECT 
    EmployeeID, 
    FirstName, 
    LastName,
    TRIM(FirstName || ' ' || LastName) AS TrimmedFullName
FROM Employees
""")

retrieve_information(cursor)

Entry found!
(1, 'Alice', 'Anderson', 'Alice Anderson')
(2, 'Bob', 'Brown', 'Bob Brown')
(3, 'Charlie', 'Clark', 'Charlie Clark')
(4, 'Diana', 'Davis', 'Diana Davis')
(5, 'Ethan', 'Evans', 'Ethan Evans')
(6, 'Fiona', 'Foster', 'Fiona Foster')
(7, 'George', 'Garcia', 'George Garcia')
(8, 'Hannah', 'Harris', 'Hannah Harris')
(9, 'Ian', 'Irwin', 'Ian Irwin')
(10, 'Julia', 'Johnson', 'Julia Johnson')
(11, 'Kevin', 'King', 'Kevin King')
(12, 'Laura', 'Lewis', 'Laura Lewis')
(13, 'Michael', 'Miller', 'Michael Miller')
(14, 'Nina', 'Nelson', 'Nina Nelson')
(15, 'Oscar', 'Owens', 'Oscar Owens')
(16, 'Paula', 'Parker', 'Paula Parker')
(17, 'Quinn', 'Quinn', 'Quinn Quinn')
(18, 'Rachel', 'Roberts', 'Rachel Roberts')
(19, 'Samuel', 'Smith', 'Samuel Smith')
(20, 'Tina', 'Taylor', 'Tina Taylor')
(21, 'Victor', 'Underwood', 'Victor Underwood')
(22, 'Wendy', 'Wilson', 'Wendy Wilson')
(23, 'Xavier', 'Young', 'Xavier Young')
(24, 'Yuki', 'Zhang', 'Yuki Zhang')
(25, 'Zach', 'White', 'Zach White')
(26, '

In [56]:
# 39. Find the most frequently ordered product.
cursor.execute("""
SELECT 
    s.ProductID, ProductName,
    COUNT(*) AS OrderCount
FROM Sales s
JOIN Products p ON s.ProductID = p.ProductID
GROUP BY s.ProductID
ORDER BY OrderCount DESC
LIMIT 1
""")

retrieve_information(cursor)

Entry found!
(7, 'Desk', 22)


In [57]:
# 40. Calculate year-over-year growth.
cursor.execute("""
SELECT 
    strftime('%Y', SaleDate) AS Year,
    SUM(Quantity * UnitPrice) AS Revenue,
    LAG(SUM(Quantity * UnitPrice)) OVER (ORDER BY strftime('%Y', SaleDate)) AS PreviousYearRevenue,
    (SUM(Quantity * UnitPrice) - LAG(SUM(Quantity * UnitPrice)) OVER (ORDER BY strftime('%Y', SaleDate))) / 
    LAG(SUM(Quantity * UnitPrice)) OVER (ORDER BY strftime('%Y', SaleDate)) * 100 AS YoYGrowth
FROM Sales
GROUP BY Year
""")

retrieve_information(cursor)

Entry found!
('2026', 86810.0, None, None)


In [58]:
# 41. Find the moving average of sales (3-day window).
cursor.execute("""
SELECT 
    SaleDate, 
    SUM(Quantity * UnitPrice) AS DailyRevenue,
    AVG(SUM(Quantity * UnitPrice)) OVER (
        ORDER BY SaleDate 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS MovingAverage
FROM Sales
GROUP BY SaleDate
""")

retrieve_information(cursor)

Entry found!
('2026-01-06', 3500.0, 3500.0)
('2026-01-13', 30.0, 1765.0)
('2026-01-17', 2500.0, 2010.0)
('2026-01-24', 210.0, 913.3333333333334)
('2026-02-02', 6200.0, 2970.0)
('2026-02-08', 1500.0, 2636.6666666666665)
('2026-02-20', 40.0, 2580.0)
('2026-02-23', 30.0, 523.3333333333334)
('2026-02-25', 2400.0, 823.3333333333334)
('2026-03-01', 350.0, 926.6666666666666)
('2026-03-02', 350.0, 1033.3333333333333)
('2026-03-08', 160.0, 286.6666666666667)
('2026-03-09', 210.0, 240.0)
('2026-03-14', 140.0, 170.0)
('2026-03-19', 450.0, 266.6666666666667)
('2026-03-24', 350.0, 313.3333333333333)
('2026-03-26', 700.0, 500.0)
('2026-03-27', 2000.0, 1016.6666666666666)
('2026-03-28', 2000.0, 1566.6666666666667)
('2026-04-03', 160.0, 1386.6666666666667)
('2026-04-04', 500.0, 886.6666666666666)
('2026-04-05', 500.0, 386.6666666666667)
('2026-04-07', 40.0, 346.6666666666667)
('2026-04-08', 150.0, 230.0)
('2026-04-09', 2480.0, 890.0)
('2026-04-14', 750.0, 1126.6666666666667)
('2026-04-17', 210.0, 1146

In [59]:
# 42. Find the difference between two dates in days.
cursor.execute("""
SELECT 
    EmployeeID, 
    HireDate,
    julianday('now') - julianday(HireDate) AS DaysSinceHire
FROM Employees
""")

retrieve_information(cursor)

Entry found!
(1, '2018-04-13', 3104.433705601841)
(2, '2019-07-26', 2635.433705601841)
(3, '2018-12-23', 2850.433705601841)
(4, '2019-01-19', 2823.433705601841)
(5, '2018-04-19', 3098.433705601841)
(6, '2022-09-15', 1488.4337056018412)
(7, '2018-01-27', 3180.433705601841)
(8, '2021-10-25', 1813.4337056018412)
(9, '2019-02-23', 2788.433705601841)
(10, '2021-11-09', 1798.4337056018412)
(11, '2026-03-08', 218.4337056018412)
(12, '2018-11-19', 2884.433705601841)
(13, '2022-01-21', 1725.4337056018412)
(14, '2018-07-07', 3019.433705601841)
(15, '2018-11-23', 2880.433705601841)
(16, '2022-04-06', 1650.4337056018412)
(17, '2019-10-29', 2540.433705601841)
(18, '2025-11-14', 332.4337056018412)
(19, '2024-10-31', 711.4337056018412)
(20, '2019-11-01', 2537.433705601841)
(21, '2020-06-18', 2307.433705601841)
(22, '2020-07-27', 2268.433705601841)
(23, '2022-07-02', 1563.4337056018412)
(24, '2026-01-19', 266.4337056018412)
(25, '2022-06-09', 1586.4337056018412)
(26, '2020-12-20', 2122.433705601841)
(

In [60]:
# 43. Find overlapping date ranges.
cursor.execute("""
SELECT 
    dr1.EmployeeID AS Employee1,
    dr2.EmployeeID AS Employee2,
    dr1.StartDate AS Start1,
    dr1.EndDate AS End1,
    dr2.StartDate AS Start2,
    dr2.EndDate AS End2
FROM DateRanges dr1
JOIN DateRanges dr2 ON dr1.EmployeeID < dr2.EmployeeID
WHERE dr1.StartDate <= dr2.EndDate AND dr2.StartDate <= dr1.EndDate
""")

retrieve_information(cursor)

Entry found!
(1, 2, '2026-01-01', '2026-03-31', '2026-02-15', '2026-05-15')
(1, 8, '2026-01-01', '2026-03-31', '2026-03-01', '2026-04-30')
(2, 3, '2026-02-15', '2026-05-15', '2026-04-01', '2026-06-30')
(2, 4, '2026-02-15', '2026-05-15', '2026-05-01', '2026-08-31')
(2, 8, '2026-02-15', '2026-05-15', '2026-03-01', '2026-04-30')
(3, 4, '2026-04-01', '2026-06-30', '2026-05-01', '2026-08-31')
(3, 8, '2026-04-01', '2026-06-30', '2026-03-01', '2026-04-30')
(4, 5, '2026-05-01', '2026-08-31', '2026-07-01', '2026-09-30')
(4, 6, '2026-05-01', '2026-08-31', '2026-08-15', '2026-10-31')
(5, 6, '2026-07-01', '2026-09-30', '2026-08-15', '2026-10-31')
(5, 7, '2026-07-01', '2026-09-30', '2026-09-01', '2026-12-31')
(6, 7, '2026-08-15', '2026-10-31', '2026-09-01', '2026-12-31')


In [61]:
# 44. Find the second most frequent salary.
cursor.execute("""
SELECT Salary, COUNT(*) AS Frequency
FROM Employees
GROUP BY Salary
ORDER BY Frequency DESC
LIMIT 1 OFFSET 1
""")

retrieve_information(cursor)

Entry found!
(48000.0, 5)


In [62]:
# 45. Find employees who earn between 40k and 60k.
cursor.execute("""
SELECT EmployeeID, FirstName, LastName, Salary
FROM Employees
WHERE Salary BETWEEN 40000 AND 60000
""")

retrieve_information(cursor)

Entry found!
(1, 'Alice', 'Anderson', 50000.0)
(2, 'Bob', 'Brown', 60000.0)
(4, 'Diana', 'Davis', 42000.0)
(11, 'Kevin', 'King', 45000.0)
(14, 'Nina', 'Nelson', 48000.0)
(16, 'Paula', 'Parker', 50000.0)
(19, 'Samuel', 'Smith', 48000.0)
(20, 'Tina', 'Taylor', 60000.0)
(22, 'Wendy', 'Wilson', 45000.0)
(26, 'Alice', 'Anderson', 52000.0)
(29, 'Diana', 'Davis', 52000.0)
(30, 'Ethan', 'Evans', 45000.0)
(33, 'Hannah', 'Harris', 42000.0)
(35, 'Julia', 'Johnson', 50000.0)
(36, 'Kevin', 'King', 42000.0)
(39, 'Nina', 'Nelson', 52000.0)
(41, 'Paula', 'Parker', 42000.0)
(43, 'Rachel', 'Roberts', 48000.0)
(44, 'Samuel', 'Smith', 48000.0)
(47, 'Wendy', 'Wilson', 58000.0)
(49, 'Yuki', 'Zhang', 48000.0)


In [63]:
# 46. Delete records having duplicate values based on one column.
cursor.execute("""
DELETE FROM Employees WHERE EmployeeID NOT IN (
    SELECT MAX(EmployeeID) FROM Employees GROUP BY Salary
)
""")

retrieve_information(cursor)

Not found


In [64]:
# 47. Insert only distinct records from another table.
cursor.execute("""
CREATE TABLE IF NOT EXISTS Employees_Distinct AS
SELECT DISTINCT * FROM Employees WHERE 1=0; -- Create an empty table with the same structure
""")

cursor.execute("""
INSERT INTO Employees_Distinct
SELECT DISTINCT * FROM Employees
""")

cursor.execute("""SELECT * FROM Employees_Distinct
""")

retrieve_information(cursor)

Entry found!
(12, 'Laura', 'Lewis', 6, 6, 72000.0, '2018-11-19', '1987-05-27', 'laura.lewis12@company.com')
(17, 'Quinn', 'Quinn', 5, 5, 70000.0, '2019-10-29', '1981-06-07', 'quinn.quinn17@company.com')
(18, 'Rachel', 'Roberts', 6, 6, 62000.0, '2025-11-14', '1999-11-21', 'rachel.roberts18@company.com')
(20, 'Tina', 'Taylor', None, 2, 60000.0, '2019-11-01', '1984-07-09', 'tina.taylor20@company.com')
(23, 'Xavier', 'Young', 5, 5, 68000.0, '2022-07-02', '1978-02-07', 'xavier.young23@company.com')
(30, 'Ethan', 'Evans', None, 6, 45000.0, '2019-03-26', '1974-11-06', 'ethan.evans30@company.com')
(31, 'Fiona', 'Foster', 1, 1, 75000.0, '2024-09-08', '1972-07-13', 'fiona.foster31@company.com')
(34, 'Ian', 'Irwin', 4, 4, 95000.0, '2020-12-28', '1994-11-11', 'ian.irwin34@company.com')
(35, 'Julia', 'Johnson', 5, 5, 50000.0, '2021-04-17', '1983-03-15', 'julia.johnson35@company.com')
(39, 'Nina', 'Nelson', 3, 3, 52000.0, '2022-03-12', '1994-03-18', 'nina.nelson39@company.com')
(40, 'Oscar', 'Owens'

In [65]:
# 48. Update salary by 10% for a specific department.
cursor.execute("""
UPDATE Employees
SET Salary = Salary * 1.10
WHERE DepartmentID = 1
""")

cursor.execute("""SELECT EmployeeID, FirstName, LastName, DepartmentID, Salary
FROM Employees
WHERE DepartmentID = 1
""")

retrieve_information(cursor)

Entry found!
(31, 'Fiona', 'Foster', 1, 82500.0)
(49, 'Yuki', 'Zhang', 1, 52800.00000000001)


In [67]:
# 49. Find employee names in UPPERCASE and lowercase.
cursor.execute("""
SELECT 
    EmployeeID, 
    UPPER(FirstName || ' ' || LastName) AS UppercaseName,
    LOWER(FirstName || ' ' || LastName) AS LowercaseName
FROM Employees
""")

retrieve_information(cursor)

Entry found!
(12, 'LAURA LEWIS', 'laura lewis')
(17, 'QUINN QUINN', 'quinn quinn')
(18, 'RACHEL ROBERTS', 'rachel roberts')
(20, 'TINA TAYLOR', 'tina taylor')
(23, 'XAVIER YOUNG', 'xavier young')
(30, 'ETHAN EVANS', 'ethan evans')
(31, 'FIONA FOSTER', 'fiona foster')
(34, 'IAN IRWIN', 'ian irwin')
(35, 'JULIA JOHNSON', 'julia johnson')
(39, 'NINA NELSON', 'nina nelson')
(40, 'OSCAR OWENS', 'oscar owens')
(41, 'PAULA PARKER', 'paula parker')
(42, 'QUINN QUINN', 'quinn quinn')
(45, 'TINA TAYLOR', 'tina taylor')
(46, 'VICTOR UNDERWOOD', 'victor underwood')
(47, 'WENDY WILSON', 'wendy wilson')
(48, 'XAVIER YOUNG', 'xavier young')
(49, 'YUKI ZHANG', 'yuki zhang')
(50, 'ZACH WHITE', 'zach white')


In [68]:
# 50. Generate a series of dates between two given dates.
cursor.execute("""
SELECT date('2026-01-01', '+' || n || ' days') AS GeneratedDate
FROM (
    SELECT 0 AS n UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4
    UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8 UNION ALL SELECT 9
    UNION ALL SELECT 10 UNION ALL SELECT 11 UNION ALL SELECT 12 UNION ALL SELECT 13 UNION ALL SELECT 14
    UNION ALL SELECT 15 UNION ALL SELECT 16 UNION ALL SELECT 17 UNION ALL SELECT 18 UNION ALL SELECT 19
    UNION ALL SELECT 20 UNION ALL SELECT 21 UNION ALL SELECT 22 UNION ALL SELECT 23 UNION ALL SELECT 24
    -- Add more as needed for longer ranges
) AS Numbers
WHERE date('2026-01-01', '+' || n || ' days') <= '2026-01-31'
""")

retrieve_information(cursor)

Entry found!
('2026-01-01',)
('2026-01-02',)
('2026-01-03',)
('2026-01-04',)
('2026-01-05',)
('2026-01-06',)
('2026-01-07',)
('2026-01-08',)
('2026-01-09',)
('2026-01-10',)
('2026-01-11',)
('2026-01-12',)
('2026-01-13',)
('2026-01-14',)
('2026-01-15',)
('2026-01-16',)
('2026-01-17',)
('2026-01-18',)
('2026-01-19',)
('2026-01-20',)
('2026-01-21',)
('2026-01-22',)
('2026-01-23',)
('2026-01-24',)
('2026-01-25',)
